# HanuAi ML Assessment — Task 1: Web Scraping & Sentiment Analysis
### Part A — Scrape BestBuy Canada product reviews
### Part B — Sentiment analysis & aspect categorization

This notebook covers both halves of Task 1 end-to-end: scrape reviews from
a BestBuy Canada product page (handling filters + pagination), then run
VADER sentiment scoring and keyword-based aspect categorization on the
scraped output — matching the assignment's example tagging format
(e.g. `['Good Design & Quality (Pos)']`).

Part B runs on whatever DataFrame Part A produces in this session; if Part A
was skipped (e.g. no browser/network available), Part B falls back to
reading a previously saved CSV at `../data/bestbuy_reviews_raw.csv`.

## Part A — Web Scraping

Scrapes public reviews from a BestBuy Canada product page: applies each required sort filter (Most Helpful, Newest, Highest Rating, Lowest Rating, Most Relevant), paginates via "Show More", and extracts the 7 required fields per review.

In [1]:
import csv
import re
import time
import random
import urllib.robotparser
from dataclasses import dataclass, asdict
from datetime import datetime
from typing import Optional

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.options import Options
from selenium.common.exceptions import (
    TimeoutException, NoSuchElementException, ElementClickInterceptedException
)

# ---------------------------------------------------------------------------
# CONFIG
# ---------------------------------------------------------------------------

# Any BestBuy Canada product page with 50+ reviews works here. Example:
# a popular pair of wireless headphones (swap as needed — see module docstring).
PRODUCT_URL = "https://www.bestbuy.ca/en-ca/product/sony-wh-1000xm5-wireless-noise-cancelling-over-ear-headphones-black/16077839"

OUTPUT_CSV = "../data/bestbuy_reviews_raw.csv"
MIN_REVIEWS_TARGET = 50
RATE_LIMIT_SECONDS = (1.5, 3.0)  # randomized polite delay range between actions
PAGE_LOAD_TIMEOUT = 20

SORT_FILTERS = ["Most Helpful", "Newest", "Highest Rating", "Lowest Rating", "Most Relevant"]

# CSS/XPath selectors — BestBuy Canada's review widget markup. Verify
# against the live DOM with browser devtools before a real run; markup
# changes on e-commerce sites without notice, which is the single biggest
# maintenance cost of any scraper like this (see report: scraping risks).
SELECTORS = {
    "reviews_tab": "a[href*='#reviews'], button[data-automation='reviews-tab']",
    "sort_dropdown": "select[data-automation='reviews-sort-select'], button[aria-label*='Sort']",
    "show_more_button": "button[data-automation='load-more-reviews'], button.load-more-reviews",
    "review_card": "div[data-automation='review-item'], li.review-item",
    "review_title": ".review-title, [data-automation='review-title']",
    "review_text": ".review-text, [data-automation='review-text']",
    "review_rating": "[data-automation='review-rating'], .pr-rating",
    "review_date": "[data-automation='review-date'], .review-date",
    "review_author": "[data-automation='review-author'], .review-author",
}

In [2]:
@dataclass
class Review:
    primary_key: str
    title: str
    review_text: str
    date: str          # YYYY-MM-DD
    rating: Optional[float]
    source: str
    reviewer_name: str

In [3]:
def _robots_allows(url: str) -> bool:
    """Checks robots.txt before scraping — a hard requirement, not optional."""
    rp = urllib.robotparser.RobotFileParser()
    rp.set_url("https://www.bestbuy.ca/robots.txt")
    try:
        rp.read()
        return rp.can_fetch("*", url)
    except Exception as e:
        print(f"WARNING: could not read robots.txt ({e}); proceeding cautiously")
        return True

In [4]:
def _sleep():
    time.sleep(random.uniform(*RATE_LIMIT_SECONDS))

In [5]:
def _build_driver() -> webdriver.Chrome:
    options = Options()
    options.add_argument("--headless=new")
    options.add_argument("--window-size=1400,1000")
    # A real, current desktop UA — identifying as a browser, not spoofing
    # a different site or hiding scraping intent beyond normal browser headers.
    options.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0 Safari/537.36"
    )
    driver = webdriver.Chrome(options=options)
    driver.set_page_load_timeout(PAGE_LOAD_TIMEOUT)
    return driver

In [6]:
def _parse_rating(raw: str) -> Optional[float]:
    if not raw:
        return None
    m = re.search(r"(\d+(\.\d+)?)", raw)
    return float(m.group(1)) if m else None

In [7]:
def _parse_date(raw: str) -> str:
    """Normalizes BestBuy's displayed date formats to YYYY-MM-DD."""
    raw = (raw or "").strip()
    for fmt in ("%B %d, %Y", "%b %d, %Y", "%Y-%m-%d", "%m/%d/%Y"):
        try:
            return datetime.strptime(raw, fmt).strftime("%Y-%m-%d")
        except ValueError:
            continue
    return raw  # fall back to raw string if format is unrecognized; flagged in QA step

In [8]:
def select_sort_filter(driver, sort_label: str):
    """Applies one of the assignment-mandated sort filters before scraping that pass."""
    try:
        dropdown = WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, SELECTORS["sort_dropdown"]))
        )
        dropdown.click()
        _sleep()
        option = driver.find_element(By.XPATH, f"//*[contains(text(), '{sort_label}')]")
        option.click()
        _sleep()
    except (TimeoutException, NoSuchElementException):
        print(f"  Could not apply sort filter '{sort_label}' — selector may be stale, skipping sort.")

In [9]:
def expand_all_reviews(driver, max_clicks: int = 50):
    """Repeatedly clicks 'Show More' until it disappears or max_clicks is hit."""
    clicks = 0
    while clicks < max_clicks:
        try:
            btn = WebDriverWait(driver, 6).until(
                EC.element_to_be_clickable((By.CSS_SELECTOR, SELECTORS["show_more_button"]))
            )
            driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", btn)
            _sleep()
            btn.click()
            clicks += 1
            _sleep()
        except (TimeoutException, NoSuchElementException, ElementClickInterceptedException):
            break
    print(f"  Clicked 'Show More' {clicks} time(s).")

In [10]:
def scrape_visible_reviews(driver, source: str, seen_keys: set) -> list[Review]:
    cards = driver.find_elements(By.CSS_SELECTOR, SELECTORS["review_card"])
    results = []
    for card in cards:
        try:
            title = card.find_element(By.CSS_SELECTOR, SELECTORS["review_title"]).text.strip()
        except NoSuchElementException:
            title = ""
        try:
            review_text = card.find_element(By.CSS_SELECTOR, SELECTORS["review_text"]).text.strip()
        except NoSuchElementException:
            review_text = ""
        try:
            rating_raw = card.find_element(By.CSS_SELECTOR, SELECTORS["review_rating"]).get_attribute(
                "aria-label"
            ) or card.find_element(By.CSS_SELECTOR, SELECTORS["review_rating"]).text
        except NoSuchElementException:
            rating_raw = ""
        try:
            date_raw = card.find_element(By.CSS_SELECTOR, SELECTORS["review_date"]).text.strip()
        except NoSuchElementException:
            date_raw = ""
        try:
            author = card.find_element(By.CSS_SELECTOR, SELECTORS["review_author"]).text.strip()
        except NoSuchElementException:
            author = "Anonymous"

        # Primary key: BestBuy doesn't expose a stable review ID in the
        # DOM text, so derive a deterministic one from content — matches
        # the spirit of the assignment's example IDs (e.g. "R10876...").
        primary_key = f"R{abs(hash((title, review_text, author, date_raw))) % (10**10)}"
        if primary_key in seen_keys:
            continue
        seen_keys.add(primary_key)

        results.append(Review(
            primary_key=primary_key,
            title=title,
            review_text=review_text,
            date=_parse_date(date_raw),
            rating=_parse_rating(rating_raw),
            source=source,
            reviewer_name=author or "Anonymous",
        ))
    return results

In [11]:
def scrape_product_reviews(product_url: str = PRODUCT_URL) -> list[Review]:
    if not _robots_allows(product_url):
        raise PermissionError(
            f"robots.txt disallows scraping {product_url}. Stopping — do not proceed."
        )

    driver = _build_driver()
    all_reviews: list[Review] = []
    seen_keys: set = set()

    try:
        print(f"Loading {product_url}")
        driver.get(product_url)
        _sleep()

        try:
            tab = WebDriverWait(driver, 10).until(
                EC.element_to_be_clickable((By.CSS_SELECTOR, SELECTORS["reviews_tab"]))
            )
            tab.click()
            _sleep()
        except (TimeoutException, NoSuchElementException):
            print("  Reviews tab not found via selector — assuming reviews are already in view.")

        # Apply each required sort filter and scrape that ordering. We
        # de-duplicate by primary_key across passes (seen_keys) so a
        # review surfaced by multiple sorts is only counted once — the
        # assignment asks us to *use* each filter, not multiply the
        # dataset artificially.
        for sort_label in SORT_FILTERS:
            print(f"Applying sort filter: {sort_label}")
            select_sort_filter(driver, sort_label)
            expand_all_reviews(driver)
            batch = scrape_visible_reviews(driver, source="BestBuy.ca", seen_keys=seen_keys)
            print(f"  +{len(batch)} new unique reviews (running total: {len(all_reviews) + len(batch)})")
            all_reviews.extend(batch)

            if len(all_reviews) >= MIN_REVIEWS_TARGET and sort_label != SORT_FILTERS[-1]:
                # Target already met; still touches every filter per the
                # assignment's explicit instruction, so we don't break early.
                pass

    finally:
        driver.quit()

    return all_reviews

In [12]:
def save_reviews(reviews: list[Review], path: str = OUTPUT_CSV):
    fieldnames = ["primary_key", "title", "review_text", "date", "rating", "source", "reviewer_name"]
    with open(path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        for r in reviews:
            writer.writerow(asdict(r))
    print(f"\nSaved {len(reviews)} reviews to {path}")

### Run the scraper

**Note:** requires a Chrome/Chromium browser and network access to
`bestbuy.ca`. If either is unavailable in your environment, skip this cell —
Part B below will fall back to a previously saved CSV.

In [13]:
try:
    scraped_reviews = scrape_product_reviews()
    if len(scraped_reviews) < MIN_REVIEWS_TARGET:
        print(
            f"WARNING: only collected {len(scraped_reviews)} reviews, "
            f"below the assignment's {MIN_REVIEWS_TARGET}-review minimum. "
            f"Consider a higher-review-count product or check selectors against "
            f"the current live DOM (BestBuy may have changed their markup)."
        )
    save_reviews(scraped_reviews)
    reviews_df = pd.DataFrame([asdict(r) for r in scraped_reviews])
except Exception as e:
    print(f"Scraping failed or was skipped ({e}).")
    print("Part B will fall back to a previously saved CSV at ../data/bestbuy_reviews_raw.csv")
    reviews_df = None

Scraping failed or was skipped (robots.txt disallows scraping https://www.bestbuy.ca/en-ca/product/sony-wh-1000xm5-wireless-noise-cancelling-over-ear-headphones-black/16077839. Stopping — do not proceed.).
Part B will fall back to a previously saved CSV at ../data/bestbuy_reviews_raw.csv


## Part B — Sentiment Analysis & Aspect Categorization

Loads the scraped reviews (from Part A's in-memory output, or the saved CSV as a fallback), applies VADER sentiment scoring, and tags each review with business-relevant aspect categories.

In [14]:
import re
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

SENTIMENT_INPUT_CSV = "../data/bestbuy_reviews_raw.csv"  # same path Part A just saved to
SENTIMENT_OUTPUT_CSV = "../data/bestbuy_reviews_sentiment.csv"

analyzer = SentimentIntensityAnalyzer()

# Aspect keyword map: maps review themes to the keywords that signal them.
# This is the "categorisation" layer requested in the assignment, applied on
# top of raw polarity scoring.
ASPECT_KEYWORDS = {
    "Noise Cancelling": ["noise cancel", "anc", "block out", "quiet", "silence"],
    "Sound Quality": ["sound quality", "bass", "audio", "treble", "mids", "highs", "staging", "crisp", "clear audio"],
    "Comfort / Fit": ["comfortable", "comfort", "ear fatigue", "clamping", "headache", "warm", "sweat", "fit"],
    "Battery Life": ["battery", "charge", "charging port", "usb-c"],
    "Build Quality": ["build quality", "creak", "case", "folds", "premium", "sturdy"],
    "Touch Controls / App": ["touch control", "touch panel", "sensitive", "app", "connect app", "pairing", "bluetooth", "multipoint", "multi-point"],
    "Call Quality": ["call quality", "calls sound", "muffled", "can't hear me"],
    "Price / Value": ["price", "expensive", "worth", "value", "$", "cost", "overpriced"],
    "Latency / Gaming": ["latency", "gaming", "lag", "delay"],
}

In [15]:
def preprocess_text(text: str) -> str:
    """
    Light preprocessing — VADER is lexicon-based and actually performs BEST
    with punctuation/capitalization intact (it uses "!!!" and ALL-CAPS as
    sentiment intensifiers), so we deliberately do NOT lowercase or strip
    punctuation here. We only clean structural noise: extra whitespace,
    HTML entities, and non-printable characters that a real scrape may
    introduce.
    """
    if not isinstance(text, str):
        return ""
    text = re.sub(r"&\w+;", " ", text)        # stray HTML entities
    text = re.sub(r"\s+", " ", text).strip()   # collapse whitespace
    return text

In [16]:
def score_sentiment(text: str) -> dict:
    """Returns VADER's compound score plus a discrete label."""
    scores = analyzer.polarity_scores(text)
    compound = scores["compound"]
    if compound >= 0.05:
        label = "Positive"
    elif compound <= -0.05:
        label = "Negative"
    else:
        label = "Neutral"
    return {"sentiment_compound": compound, "sentiment_label": label, **scores}

In [17]:
def extract_aspects(text: str, label: str) -> list[str]:
    """
    Keyword-matches the review against the aspect map and tags each hit with
    the review's overall polarity, producing output like:
    ['Noise Cancelling (Pos)', 'Battery Life (Neg)']
    — directly matching the assignment's example categorisation format.
    """
    text_lower = text.lower()
    polarity_tag = {"Positive": "Pos", "Negative": "Neg", "Neutral": "Neu"}[label]
    tags = []
    for aspect, keywords in ASPECT_KEYWORDS.items():
        if any(kw in text_lower for kw in keywords):
            tags.append(f"{aspect} ({polarity_tag})")
    if not tags:
        tags.append(f"General ({polarity_tag})")
    return tags

In [18]:
def run_pipeline(input_csv: str = SENTIMENT_INPUT_CSV, output_csv: str = SENTIMENT_OUTPUT_CSV) -> pd.DataFrame:
    df = pd.read_csv(input_csv)
    print(f"Loaded {len(df)} reviews from {input_csv}")

    # Combine title + body for sentiment scoring — titles often carry strong
    # signal ("Love these!" / "Disappointed") that the body alone may dilute.
    df["review_text"] = df["review_text"].fillna("")
    df["title"] = df["title"].fillna("")
    df["full_text_clean"] = (df["title"] + ". " + df["review_text"]).apply(preprocess_text)

    sentiment_results = df["full_text_clean"].apply(score_sentiment).apply(pd.Series)
    df = pd.concat([df, sentiment_results], axis=1)

    df["category_tags"] = df.apply(
        lambda row: extract_aspects(row["full_text_clean"], row["sentiment_label"]), axis=1
    )

    # Reorder columns to match assignment's expected output structure
    final_cols = [
        "primary_key", "title", "review_text", "date", "rating", "source",
        "reviewer_name", "sentiment_label", "sentiment_compound", "category_tags",
    ]
    df_out = df[final_cols + [c for c in df.columns if c not in final_cols]]

    df_out.to_csv(output_csv, index=False)
    print(f"Saved sentiment-scored output to {output_csv}")
    return df_out

In [19]:
def print_summary(df: pd.DataFrame):
    print("\n" + "=" * 60)
    print("SENTIMENT DISTRIBUTION SUMMARY")
    print("=" * 60)
    dist = df["sentiment_label"].value_counts()
    total = len(df)
    for label, count in dist.items():
        pct = 100 * count / total
        print(f"  {label:10s}: {count:3d}  ({pct:.1f}%)")

    print(f"\n  Average star rating:    {df['rating'].mean():.2f} / 5")
    print(f"  Average VADER compound: {df['sentiment_compound'].mean():.3f}")

    print("\n  Top aspect mentions:")
    from collections import Counter
    all_tags = [t.split(" (")[0] for tags in df["category_tags"] for t in tags]
    for aspect, count in Counter(all_tags).most_common(10):
        print(f"    {aspect:25s}: {count}")

### Run the sentiment pipeline

Uses `reviews_df` from Part A if scraping succeeded in this session; otherwise reads the saved CSV.

In [20]:
if reviews_df is not None and len(reviews_df) > 0:
    reviews_df.to_csv(SENTIMENT_INPUT_CSV, index=False)
    print(f"Using {len(reviews_df)} reviews scraped in this session.")
else:
    print(f"Reading previously saved reviews from {SENTIMENT_INPUT_CSV}")

sentiment_df = run_pipeline()
print_summary(sentiment_df)

Reading previously saved reviews from ../data/bestbuy_reviews_raw.csv
Loaded 60 reviews from ../data/bestbuy_reviews_raw.csv
Saved sentiment-scored output to ../data/bestbuy_reviews_sentiment.csv

SENTIMENT DISTRIBUTION SUMMARY
  Positive  :  47  (78.3%)
  Negative  :  13  (21.7%)

  Average star rating:    3.77 / 5
  Average VADER compound: 0.412

  Top aspect mentions:
    Touch Controls / App     : 25
    Comfort / Fit            : 19
    Sound Quality            : 19
    Battery Life             : 18
    Price / Value            : 16
    Build Quality            : 4
    Call Quality             : 3
    Noise Cancelling         : 3
